# Gold Setup Step 5 — Build Category Embeddings
Computes one centroid vector per category from labeled products.
This is the **memory** of the system — run once after notebook 02, then re-run after each review session in notebook 07.

**Output:** `workspace.superapp.gold_category_centroids`

In [0]:
# Sentence-transformers should already be installed
# If not, install with: %pip install sentence-transformers --quiet
print('✅ Package check complete')

In [0]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
import json
from pyspark.sql.functions import col

print('Libraries loaded.')

In [0]:
# Load all classified products with their names from silver.
# Use subcategory if exists, otherwise use category (COALESCE).
labeled = spark.sql("""
    SELECT DISTINCT
        pc.id_producto,
        COALESCE(pc.id_subcategoria, pc.id_categoria) AS id_categoria_final,
        COALESCE(gcs.nombre, gc.nombre) AS categoria_nombre,
        sp.producto
    FROM workspace.superapp.gold_productos_categorias pc
    LEFT JOIN workspace.superapp.gold_categorias gc
        ON pc.id_categoria = gc.id_categoria
    LEFT JOIN workspace.superapp.gold_categorias gcs
        ON pc.id_subcategoria = gcs.id_categoria
    JOIN workspace.superapp.silver_prices sp
        ON pc.id_producto = sp.id_producto
    WHERE COALESCE(pc.id_subcategoria, pc.id_categoria) IS NOT NULL
      AND COALESCE(gcs.nombre, gc.nombre) != 'sin_clasificar'
      AND COALESCE(gcs.nombre, gc.nombre) NOT LIKE '\_OLD\_%'
      AND COALESCE(gcs.nombre, gc.nombre) NOT LIKE '\_DELETED\_%'
      AND sp.producto IS NOT NULL
""").toPandas()

print(f'Labeled products loaded:  {len(labeled):,}')
print(f'Categories represented:   {labeled["categoria_nombre"].nunique()}')
print()
print(labeled.groupby('categoria_nombre').size().sort_values(ascending=False).head(50).to_string())

In [0]:
# Embed all labeled product names.
# CRITICAL: Must use same model as convergence loop (intfloat/multilingual-e5-small)
model = SentenceTransformer('intfloat/multilingual-e5-small')

print(f'Generating embeddings for {len(labeled):,} products...')
embeddings = model.encode(
    labeled['producto'].tolist(),
    show_progress_bar=True,
    batch_size=128
)
labeled['embedding'] = list(embeddings)
print(f'Done. Embedding dim: {embeddings.shape[1]}')

In [0]:
# Centroid = mean of member embeddings, L2-normalized.
# Normalization makes cosine similarity == dot product (faster at query time).
print('Computing centroids...')
centroids = []

for cat_nombre, group in labeled.groupby('categoria_nombre'):
    vecs = np.array(group['embedding'].tolist())
    centroid = vecs.mean(axis=0)
    norm = np.linalg.norm(centroid)
    if norm > 1e-10:
        centroid = centroid / norm
    centroids.append({
        'id_categoria':  int(group['id_categoria_final'].iloc[0]),  # Use id_categoria_final
        'nombre':        cat_nombre,
        'n_productos':   int(len(group)),
        'centroid_json': json.dumps(centroid.tolist()),
    })

centroids_pd = pd.DataFrame(centroids)
print(f'Centroids computed: {len(centroids_pd)}')
print()
print(centroids_pd[['nombre', 'n_productos']].sort_values('n_productos', ascending=False).head(50).to_string())

In [0]:
# Persist — overwrite so each run refreshes the snapshot.
(spark.createDataFrame(centroids_pd[['id_categoria', 'nombre', 'n_productos', 'centroid_json']])
      .write.mode('overwrite')
      .option('overwriteSchema', 'true')
      .saveAsTable('workspace.superapp.gold_category_centroids'))

print(f'Saved {len(centroids_pd)} centroids to workspace.superapp.gold_category_centroids')

In [0]:
%sql
-- 1. Merge duplicate: Toallas Femeninas → Toalla Femenina (keep ID 52)
UPDATE workspace.superapp.gold_productos_categorias
SET id_categoria = 52, id_subcategoria = NULL
WHERE id_categoria = 64 OR id_subcategoria = 64;

UPDATE workspace.superapp.gold_categorias
SET nombre = '_DELETED_Toallas Femeninas'
WHERE id_categoria = 64;

-- 2. Merge duplicate: cremas_para_peinar → Crema Peinar (keep ID 60)
UPDATE workspace.superapp.gold_productos_categorias
SET id_categoria = 60, id_subcategoria = NULL
WHERE id_categoria = 138 OR id_subcategoria = 138;

UPDATE workspace.superapp.gold_categorias
SET nombre = '_DELETED_cremas_para_peinar'
WHERE id_categoria = 138;

-- 3. Standardize lowercase categories to Title Case
UPDATE workspace.superapp.gold_categorias SET nombre = 'Aceite' WHERE id_categoria = 2;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Agua' WHERE id_categoria = 13;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Arroz' WHERE id_categoria = 1;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Azúcar' WHERE id_categoria = 6;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Café' WHERE id_categoria = 10;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Cerveza' WHERE id_categoria = 15;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Fideos' WHERE id_categoria = 3;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Galletitas' WHERE id_categoria = 8;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Gaseosa' WHERE id_categoria = 12;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Harina' WHERE id_categoria = 20;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Jabón' WHERE id_categoria = 7;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Jugo' WHERE id_categoria = 14;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Leche' WHERE id_categoria = 4;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Manteca' WHERE id_categoria = 17;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Pan' WHERE id_categoria = 9;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Queso' WHERE id_categoria = 18;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Té' WHERE id_categoria = 11;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Vino' WHERE id_categoria = 16;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Yerba' WHERE id_categoria = 5;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Yogur' WHERE id_categoria = 19;

-- 4. Standardize snake_case to Title Case
UPDATE workspace.superapp.gold_categorias SET nombre = 'Aceite en Spray' WHERE id_categoria = 113;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Agua Micelar' WHERE id_categoria = 127;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Artículos de Dibujo' WHERE id_categoria = 106;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Calefactores y Comederos' WHERE id_categoria = 149;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Chocolate con Leche' WHERE id_categoria = 114;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Chocolates' WHERE id_categoria = 109;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Condimentos y Jugos en Polvo' WHERE id_categoria = 123;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Cuadernos' WHERE id_categoria = 122;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Cuidado Bebés' WHERE id_categoria = 150;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Desodorantes' WHERE id_categoria = 110;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Edulcorantes y Suplementos en Polvo' WHERE id_categoria = 135;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Especias y Condimentos' WHERE id_categoria = 121;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Espumante' WHERE id_categoria = 112;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Fibra Esponja' WHERE id_categoria = 133;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Frutas Congeladas' WHERE id_categoria = 141;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Galletitas con Chips' WHERE id_categoria = 137;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Gaseosas Colas' WHERE id_categoria = 131;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Gaseosas Lima Limón' WHERE id_categoria = 151;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Heladeras No Frost' WHERE id_categoria = 145;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Jamón Cocido' WHERE id_categoria = 134;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Jarras Mug' WHERE id_categoria = 143;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Mermeladas y Frutas Rojas' WHERE id_categoria = 136;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Panes Especiales' WHERE id_categoria = 139;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Pañales' WHERE id_categoria = 119;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Paños' WHERE id_categoria = 116;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Platos Hondos' WHERE id_categoria = 120;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Polvos para Hornear' WHERE id_categoria = 120;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Productos Congelados' WHERE id_categoria = 117;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Productos del Mar' WHERE id_categoria = 142;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Protector Solar' WHERE id_categoria = 126;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Remeras Estampadas' WHERE id_categoria = 129;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Ropa y Accesorios' WHERE id_categoria = 118;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Ropa y Accesorios Infantiles' WHERE id_categoria = 144;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Sahumerios' WHERE id_categoria = 132;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Shampoo' WHERE id_categoria = 124;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Snacks Salados' WHERE id_categoria = 115;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Suavizantes y Aprestos' WHERE id_categoria = 148;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Tapas Pascualinas' WHERE id_categoria = 147;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Té Infusión' WHERE id_categoria = 107;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Té Verde' WHERE id_categoria = 140;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Televisores' WHERE id_categoria = 111;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Textil de Baño' WHERE id_categoria = 108;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Vinagres y Condimentos Especiales' WHERE id_categoria = 130;
UPDATE workspace.superapp.gold_categorias SET nombre = 'Yogur Bebible' WHERE id_categoria = 146;

SELECT 'Categories standardized and duplicates merged' AS status;

In [0]:
%sql
-- Show 100 sample products with their assigned categories
SELECT 
    sp.producto,
    COALESCE(gcs.nombre, gc.nombre) AS categoria_asignada,
    gc.nombre AS categoria_padre,
    gcs.nombre AS subcategoria,
    pc.metodo
FROM workspace.superapp.gold_productos_categorias pc
LEFT JOIN workspace.superapp.gold_categorias gc
    ON pc.id_categoria = gc.id_categoria
LEFT JOIN workspace.superapp.gold_categorias gcs
    ON pc.id_subcategoria = gcs.id_categoria
JOIN workspace.superapp.silver_prices sp
    ON pc.id_producto = sp.id_producto
WHERE COALESCE(gcs.nombre, gc.nombre) != 'sin_clasificar'
  AND COALESCE(gcs.nombre, gc.nombre) NOT LIKE '_DELETED%'
ORDER BY RAND()
LIMIT 100;